8711cc39b5ab4cf58b273b8fdbc6580b

In [3]:
import numpy as np
import requests
import re
from sklearn.metrics.pairwise import cosine_similarity

# API Key untuk NewsAPI (Ganti dengan API key Anda)
API_KEY = "8711cc39b5ab4cf58b273b8fdbc6580b"
NEWS_API_URL = "https://newsapi.org/v2/everything?q=technology&language=en&apiKey=" + API_KEY

# Fungsi untuk mengambil berita dari NewsAPI
def get_news_articles():
    response = requests.get(NEWS_API_URL)
    if response.status_code == 200:
        data = response.json()
        articles = [article['title'] + ' ' + article['description'] for article in data['articles'] if article['description']]
        return ' '.join(articles)
    else:
        print("Gagal mengambil data dari NewsAPI")
        return ""

class SkipGramModel:
    def __init__(self, vocab_size, embedding_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.W1 = np.random.randn(vocab_size, embedding_dim) * np.sqrt(1.0 / vocab_size)
        self.W2 = np.random.randn(embedding_dim, vocab_size) * np.sqrt(1.0 / embedding_dim)

    def forward(self, one_hot_vector):
        hidden_layer = np.dot(one_hot_vector, self.W1)
        output_layer = np.dot(hidden_layer, self.W2)
        output_layer = self._softmax(output_layer)
        return hidden_layer, output_layer

    def backward(self, one_hot_vector, target_vector, learning_rate=0.005, lambda_reg=0.0001):
        hidden_layer, output_layer = self.forward(one_hot_vector)
        error = target_vector - output_layer
        output_layer_gradient = np.outer(hidden_layer, error)
        hidden_layer_gradient = np.outer(one_hot_vector, np.dot(self.W2, error))
        output_layer_gradient = np.clip(output_layer_gradient, -1, 1)
        hidden_layer_gradient = np.clip(hidden_layer_gradient, -1, 1)
        self.W1 -= learning_rate * (hidden_layer_gradient + lambda_reg * self.W1)
        self.W2 -= learning_rate * (output_layer_gradient + lambda_reg * self.W2)

    def _softmax(self, x, epsilon=1e-10):
        exp_x = np.exp(x - np.max(x))
        return exp_x / (exp_x.sum() + epsilon)

def prepare_training_data(window_size):
    text = get_news_articles()
    words = re.findall(r'\b\w+\b', text.lower())
    vocab = list(set(words))
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    training_pairs = []
    for i, target_word in enumerate(words):
        target_idx = word2idx[target_word]
        for j in range(-window_size, window_size + 1):
            if j != 0 and 0 <= i + j < len(words):
                context_word = words[i + j]
                training_pairs.append((target_idx, word2idx[context_word]))
    return training_pairs, vocab, word2idx

def evaluate_word_similarity(model, word2idx, target_word, top_n=5):
    if target_word not in word2idx:
        return []
    target_vector = model.W1[word2idx[target_word]].reshape(1, -1)
    similarities = {}
    for word, idx in word2idx.items():
        if word != target_word:
            word_vector = model.W1[idx].reshape(1, -1)
            similarities[word] = cosine_similarity(target_vector, word_vector)[0][0]
    return sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]

def train_and_analyze(window_sizes, embedding_dims, epochs=10):
    results = []
    for window_size in window_sizes:
        for embedding_dim in embedding_dims:
            training_pairs, vocab, word2idx = prepare_training_data(window_size)
            model = SkipGramModel(vocab_size=len(vocab), embedding_dim=embedding_dim)
            total_loss = 0
            for epoch in range(epochs):
                epoch_loss = 0
                for target_idx, context_idx in training_pairs:
                    target_vector = np.zeros(len(vocab))
                    target_vector[target_idx] = 1
                    context_vector = np.zeros(len(vocab))
                    context_vector[context_idx] = 1
                    hidden, output = model.forward(target_vector)
                    loss = -np.log(output[context_idx] + 1e-10)
                    model.backward(target_vector, context_vector)
                    epoch_loss += loss
                total_loss = epoch_loss / len(training_pairs)

            similar_words = evaluate_word_similarity(model, word2idx, "technology")
            top_words = ', '.join([word for word, _ in similar_words])

            if any(word in top_words for word in ['ai', 'digital', 'innovation', 'computing', 'cloud']):
                analysis = 'Baik (Kata relevan ditemukan)'
            else:
                analysis = 'Kurang Baik (Banyak stopwords atau kurang relevan)'

            results.append((window_size, embedding_dim, total_loss, top_words, analysis))

    print("\nHasil Analisis:")
    print("| Window Size | Embedding Dim | Loss Akhir | Top-5 Similar Words | Analisis |")
    print("|-------------|---------------|------------|---------------------|----------|")
    for res in results:
        print(f"| {res[0]:<11} | {res[1]:<13} | {res[2]:<10.4f} | {res[3]:<19} | {res[4]} |")

if __name__ == "__main__":
    window_sizes = [1, 2, 3]
    embedding_dims = [50, 100, 200]
    train_and_analyze(window_sizes, embedding_dims)



Hasil Analisis:
| Window Size | Embedding Dim | Loss Akhir | Top-5 Similar Words | Analisis |
|-------------|---------------|------------|---------------------|----------|
| 1           | 50            | 7.3658     | of, world, in, at, potential | Kurang Baik (Banyak stopwords atau kurang relevan) |
| 1           | 100           | 7.3667     | of, world, verge, first, in | Kurang Baik (Banyak stopwords atau kurang relevan) |
| 1           | 200           | 7.3743     | in, world, next, at, first | Kurang Baik (Banyak stopwords atau kurang relevan) |
| 2           | 50            | 21.6384    | to, in, for, of, and | Kurang Baik (Banyak stopwords atau kurang relevan) |
| 2           | 100           | 18.1896    | of, is, in, to, s   | Kurang Baik (Banyak stopwords atau kurang relevan) |
| 2           | 200           | 19.9132    | to, in, is, s, of   | Kurang Baik (Banyak stopwords atau kurang relevan) |
| 3           | 50            | 22.8719    | is, to, s, a, and   | Kurang Baik (Ba